<a href="https://colab.research.google.com/github/Amruda-glitch/Hands-on-Training/blob/main/Working_on_RAG_using_Groq_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu sentence-transformers openai

In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.1 MB/s eta 0:00:00


In [ ]:
from groq import Groq

SentenceTransfomer --> converts into **Embedding** (Embeddings are numerical vector representations of data that capture semantic meaning and relationships.)

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
Groq.api_key ='API_KEY'

In [ ]:
docs = [
    "Deep Learning (DL) is a subset of Machine Learning that uses multi-layered neural networks to automatically learn complex patterns from large amounts of data.",
    "LSTM (Long Short-Term Memory) is a type of Recurrent Neural Network (RNN) designed to remember long-term dependencies and process sequential data effectively.",
    "BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based language model developed by Google that understands the context of words by reading text in both directions.",
    "RoBERTa (Robustly Optimized BERT Pretraining Approach) is an enhanced version of BERT developed by Meta AI, trained with optimized techniques and more data to achieve better language understanding.",
    "OpenAI is an artificial intelligence research and technology company that develops advanced AI models, including the GPT family, for tasks such as conversation, coding, summarization, and content generation.",
    "Transfer Learning is a technique where knowledge from a pre-trained model is reused for a new but related task.",
    "Prompt Engineering is the practice of designing effective prompts to guide AI models toward desired outputs.",
    "Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with language generation to produce more accurate and up-to-date responses.Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with language generation to produce more accurate and up-to-date responses.",
    "Embeddings are numerical vector representations of data that capture semantic meaning and relationships."
]

In [ ]:
model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
doc_embed= model.encode(docs).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
doc_embed.shape

(9, 384)

The first line mentions the size of the vectores (384) , secondly index stores data inside faiss , includes  **euclidean distance** between the similarity btw the words (L2) and IndexFlat --> returns data as it is and storing into the fiass. Thirdly, with the similarity calculated using Euclidean dist. is added along with the index (in which data stores in faiss)

In [ ]:
dim=doc_embed.shape[1]
index=faiss.IndexFlatL2(dim)
index.add(doc_embed)

In [ ]:
qry="What is the Transfer Learning?"
qry_embed=model.encode([qry]).astype('float32')

top_k=2
distances, indices = index.search(qry_embed, top_k)

retrieved_chunks = [docs[i] for i in indices[0]]
print('Retrieved chunks:')
for chunk in retrieved_chunks:
  print('-', chunk)


Retrieved chunks:
- Transfer Learning is a technique where knowledge from a pre-trained model is reused for a new but related task.
- Deep Learning (DL) is a subset of Machine Learning that uses multi-layered neural networks to automatically learn complex patterns from large amounts of data.


In [ ]:
client = Groq(api_key="API_KEY")

In [ ]:
context = "\n".join(retrieved_chunks)
prompt = f"""
Answer the qn based only on the context below.
Context:
{context}
Question:
{qry}
Answer:
"""
response = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {"role":"user","content":prompt}
    ],
    temperature=0.2
)
print("Final Answer:\n")
print(response.choices[0].message.content)

Final Answer:

Transfer Learning is a technique where knowledge from a pre-trained model is reused for a new but related task.
